# Sistema RAG – Atividade 4 (Tecnologias Emergentes)

Notebook para rodar todo o sistema no **Google Colab**: indexação, perguntas, avaliação e app de demonstração.

**Antes de começar:** `Ambiente de execução > Alterar o tipo de ambiente de execução > GPU T4` (deixa o LLM bem mais rápido; sem GPU também funciona, só que devagar).

## 1. Baixar o projeto
Se o projeto estiver no GitHub, ajuste a URL abaixo. Se não, envie o `.zip` do projeto pelo painel de arquivos do Colab e use a segunda opção.

In [ ]:
REPO_URL = "https://github.com/opittas/SymplaDocs.git"  # ajuste aqui
!git clone -q $REPO_URL projeto
# Alternativa sem GitHub: envie projeto.zip e descomente a linha abaixo
# !unzip -q projeto.zip -d projeto
%cd projeto

## 2. Instalar as dependências

In [ ]:
!pip install -q sentence-transformers chromadb pyyaml gradio accelerate

## 3. Escolher o backend do LLM
No Colab usamos o `transformers` (backend `hf`) com o modelo aberto **Qwen2.5-1.5B-Instruct**. Localmente, o padrão é o Ollama.

In [ ]:
import os
os.environ["RAG_LLM"] = "hf"
os.environ["RAG_MODELO_HF"] = "Qwen/Qwen2.5-1.5B-Instruct"

## 4. Indexar os documentos
Cria um índice no Chroma para cada tamanho de chunk testado (300, 600 e 1000 caracteres).

In [ ]:
!python indexar.py --todos

## 5. Avaliação da recuperação
Compara os 3 tamanhos de chunk com precision@k, hit@k e MRR, e sugere o limiar de similaridade. O resultado vai para `avaliacao/resultados/resultados.md`.

In [ ]:
!python avaliacao/avaliar.py

In [ ]:
from IPython.display import Markdown
Markdown(open('avaliacao/resultados/resultados.md', encoding='utf-8').read())

### (Opcional) Comparar dois modelos de embeddings

In [ ]:
# !python avaliacao/avaliar.py --embeddings sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 intfloat/multilingual-e5-small

## 6. Avaliação completa com o LLM
Gera a resposta de todas as perguntas de teste e confere se o sistema responde quando há resposta e recusa quando não há. Com GPU T4 leva poucos minutos.

Se o passo 5 sugeriu outro limiar, passe com `--limiar`.

In [ ]:
!python avaliacao/avaliar.py --gerar --llm hf

In [ ]:
Markdown(open('avaliacao/resultados/resultados.md', encoding='utf-8').read())

## 7. Fazer perguntas

In [ ]:
import sys; sys.path.insert(0, '.')
from src.rag import SistemaRAG
rag = SistemaRAG(backend_llm='hf')

def perguntar(p):
    r = rag.responder(p)
    print(r.texto)
    for t in r.trechos:
        print(f'  [{t.similaridade:.2f}] {t.citacao()}')

In [ ]:
perguntar("Escreva sua pergunta aqui")

## 8. App de demonstração (Gradio)
Gera um link público temporário para mostrar o sistema na apresentação.

In [ ]:
!python app.py --llm hf --share

## 9. Baixar os resultados
Baixe `avaliacao/resultados/` para colocar no repositório e no relatório.

In [ ]:
!zip -qr resultados.zip avaliacao/resultados
from google.colab import files
files.download('resultados.zip')